# CoDA

我们之前所提及的所有基于 Diffusion 方法，包括 GLaD, D4M, MGD3 与 DAP，实际上最适配的 Diffusion 模型都是在被蒸馏的数据集上训练的 Diffusion 模型。这很容易理解，因为 Diffusion 模型生成的图像需要作为原始数据集的蒸馏数据集出现，这意味着其生成空间最好被限制在真实数据集分布上。对于一般通用模型，蒸馏效果反而会下降。我们试图解决这个问题。

推荐你读 https://arxiv.org/pdf/2512.03844 CoDA: From Text-to-Image Diffusion Models to Training-Free Dataset Distillation  CoDA 与上一章提到的 DAP 与 MGD3 同样也是一种基于 Guidance 数据集蒸馏方法。

# 基本逻辑

我们认为，通用 T2I 生成模型生成与被蒸馏数据集之间的差距来自对于 Guidance 代表元素的选取。传统的方法使用 K-means 这样简单的聚类方法来选出核心代表元素，这未免过于草率。因为 K-means 假定了元素都呈球状凸集分布，这是一个非常强的假设，而现实中的数据分布往往并非如此。

所以，最简单直接的方法就是将聚类方法改为更复杂的聚类，比如我们即将要提到的 HDBSCAN 密度聚类。

## Distribution Discover

第一阶段是 CoDA 的方法核心。

首先我们需要解决一件事，通用 T2I 模型中的 Prompt 意思与我们想要蒸馏的真实数据集之间不一定对齐。例如 Stable Diffusion 理解的狗可能包含很多新颖的造型，这与 ImgaeNet-1K 中的也许不太一样。CoDA 为了克服这个问题，提出第一阶段先进行 Distribution Discover，简而言之就是先找出被蒸馏数据集的代表元素。

我们记类别 $c$ 真实数据为
$$\mathcal T_c=
\{x_1^c,x_2^c,\ldots,x_{N_c}^c\}.$$
首先我们计算潜空间对应
$$z_i^c=
\mathcal E_{\mathrm{VAE}}(x_i^c).$$
对潜空间内元素 $z_i^c$ 做聚类得到
$$\mathcal R_c=
\{r_{c,1},r_{c,2},\ldots,r_{c,\mathrm{IPC}}\}.$$

此处，MGD3 选择对于这些潜空间元素直接做 K-means 聚类。但是作者观察到一个问题，那就是原始 VAE Encoder 并不是为分类准备的。我们知道 VAE Encoder 将原始像素空间中元素映射到潜空间上近似高斯分布，换言之近似高斯球壳的分布
$$p(z)\approx\mathcal N(0,I).$$
VAE latent 更关心能否重建图像，而不是能否把类别分开。所以 VAE Encoder 处理之后的潜空间元素可能分离性很差，不同类别之间的元素混杂在一起。这意味着当我们处理对于潜空间内元素聚类，K-means 可能没那么管用。

我们之前提及的 MGD3 等等方法非常喜爱的 K-means 聚类，被 CoDA 认为是不合适的
$$\{z_i^c\}
\longrightarrow
\{m_{c,1},\ldots,m_{c,K}\}.$$
K-means 假设聚类类似球形凸集并且方差相近，但是 VAE latent 很有可能不均匀且扭曲。K-means 可能把一个类内真实 mode 切坏，也可能把多个类内 mode 混在一起。实际上在诸多聚类算法中，K-means 是最简单的方法，就好像优化器中的 SGD。我们需要更加高级的聚类方法来克服一些复杂场景中的聚类困难。

因此，CoDA 选择了 HDBSCAN 密度聚类。我们做以下几件事。

首先将 VAE latent 降低到更低维度，这一步我们使用 UMAP
$$z_i\in\mathbb R^d
\longrightarrow
y_i\in\mathbb R^{d'}.$$
这是为了让密度聚类更容易发现聚类结构以及减轻高维空间中元素密度分布过度不均匀问题。

我们简单说说 UMAP 算法，这是一个将高维流形降维的算法。我们定义一个权重
$$p_{j\mid i}
=
\exp
\left(
-
\frac{
\max\{0,d(z_i,z_j)-\rho_i\}
}{
\sigma_i
}
\right).$$
其中 $d(z_i,z_j)$ 是两点高维空间中距离，$\rho_i$ 是局部偏移量，指的是对于 $z_i$ 而言最近点距离
$$\rho_i=
\min_{j:\,d_{ij} \gt0}d_{ij}.$$
$\sigma_i$ 是局部尺度，需要满足一个关系
$$\sum_{j=1}^{k}
\exp
\left(
-
\frac{
\max\{0,d_{ij}-\rho_i\}
}{
\sigma_i
}
\right)
=
\log_2(k).$$
其中 $k$ 是设定的超参数。所以解出 $\sigma_i$ 是解一个方程，可以通过简单的解析计算方法找出。

总之，$p_{j\mid i}$ 表示从 $z_i$ 看，$z_j$ 属于其局部邻域的强度。对于高密度区域与低密度区域，局部尺度进行了某种意义上的归一化，这意味着这个量是相对的。

通过这个权重我们可以定义一个加权边无向图，其每边权重是
$$p_{ij}
=
p_{j\mid i}
+
p_{i\mid j}
-
p_{j\mid i}p_{i\mid j}.$$

现在我们开始给出最终的低维坐标诞生方法。首先给每点一个可学习低维坐标
$$y_i\in\mathbb R^{d'}.$$
定义低维相似度
$$q_{ij}
=
\frac{1}{
1+a\|y_i-y_j\|_2^{2b}
}.$$
其中 $a, b$ 是超参数。我们认为，高维空间中距离很大，那么低维空间中距离应该同样很大。这意味着优化对象是
$$\min_{\{\tilde z_i\}}
\operatorname{CE}
\left(
\{p_{ij}\},
\{q_{ij}\}
\right).$$
其中 $\operatorname{CE}$ 指的是交叉熵损失。换言之
$$\mathcal L_{\mathrm{UMAP}}
=
\sum_{i\lt j}
\left[
p_{ij}\log\frac{p_{ij}}{q_{ij}}
+
(1-p_{ij})\log\frac{1-p_{ij}}{1-q_{ij}}
\right].$$
需要被优化。

所以 UMAP 只是一种更加合理的降维算法。我们做聚类之前先做降维，是因为低维空间更适合观察局部密度结构。

在 UMAP 做完流形的降维之后，我们可以开始对于低维元素进行 HDBSCAN 聚类。假设我们有低维点
$$y_1,y_2,\ldots,y_N.$$
首先计算
$$\operatorname{core}_k(y_i)
=
d
\left(
y_i,
\operatorname{kNN}_k(y_i)
\right).$$
此处 $\operatorname{kNN}$ 指的是距离点的第 $k$ 邻近点，$k$ 是一个设定的超参数。需要注意，不要将 UMAP 和 HDBSCAN 的超参数 $k$ 混淆，这并不是同一个超参数。

我们定义 mutual reachability distance
$$d_{\mathrm{mreach}}(y_i,y_j)
=
\max
\left\{
\operatorname{core}_k(y_i),
\operatorname{core}_k(y_j),
d(y_i,y_j)
\right\}.$$
这个共同距离保证了稀疏区域内点即使距离相近也不会被认为是一个高密度区域。所以现在可以根据这个距离再定义一个加权无向图，每边权重就是距离。

对于这个加权无向图，我们可以生成最小生成树 $E_{\mathrm{MST}}$，指的是能够将所有点连接的最小边集合，并且我们要求其总边权重和最小
$$E_{\mathrm{MST}}
=
\arg\min_{E}
\sum_{(i,j)\in E}
d_{\mathrm{mreach}}(y_i,y_j).$$

对于这个最小生成树，我们可以进行某种意义上的切分。我们仅仅保留那些足够亲密的点，换言之，如果发生
$$d_{\mathrm{mreach}}(y_i,y_j)\gt \varepsilon$$
那么这两点被视为不够亲密，我们切断 $i, j$ 之间边。这意味着被保留的点之间连接关系天然会形成一个聚类。



请注意此处 $\epsilon$ 是变化的，换言之我们考察多个 $\epsilon$ 下的聚类情况。定义密度尺度
$$\lambda=
\frac{1}{\varepsilon}.$$

当密度尺度不断变大，原始最小生成树会被切割得越来越细。当一个聚类被切割，如果子分支包含节点个数大于阈值 $m$，我们认为其有资格成为一个聚类；反之我们认为其仅仅是从母分支中掉出，后续我们会直接忽略这个分支。这被称为 Condensed Tree，我们需要忽略那些太微小的分支。

所以，在这个切分过程中，每个子聚类都会来自一个母聚类，每个母聚类被切割都会掉落微小点与子聚类。这个过程可以被一棵树来描述，类似下图。

<img src="./assets/HDBSCAN.png" width="600" height="450">

最终，我们希望保留多个尺度下都稳定存在的聚类，这实际上是层级聚类树的思想。我们定义聚类稳定性
$$\operatorname{Stability}(C)
=
\sum_{y_i\in C}
\left(
\lambda_i^{\mathrm{leave}}
-
\lambda_C^{\mathrm{birth}}
\right).$$
其中 $\lambda_C^{\mathrm{birth}}$ 是聚类 $C$ 出现的密度尺度，$\lambda_i^{\mathrm{leave}}$ 是 $y_i$ 离开该点时的密度尺度。如果一个点的 $\lambda_i^{\mathrm{leave}}$ 很大，说明点 $y_i$ 在高密度要求下仍然属于这个聚类，是核心点，反之则是边缘点。这意味着一个聚类如果在很长密度尺度范围内包含足够多点，那么就有资格被称为是一个真正的聚类。

最后 HDBSCAN 会从 Condensed Tree 中选一组互不重叠的聚类，使总稳定性较高，换言之没有任何聚类之间存在单纯母子关系。默认策略通常称为 EOM，也就是 Excess of Mass。这些分支就是最终聚类。没有被稳定聚类接纳的点就是 noise 或 outliers。

我们目前还仅仅指出哪些聚类是稳定的，但是如何像 K-means 那样选出一个聚类代表元？实际上 HDBSCAN 会为每个聚类中元素分配一个成员权重 $p_i$
$$p_i = \frac{\lambda_i^{\mathrm{leave}}-\lambda_C^{\mathrm{birth}}}{\lambda_C^{\mathrm{max}}-\lambda_C^{\mathrm{birth}}}.
$$
其中 $\lambda_C^{\mathrm{max}}$ 是聚类 $C$ 内点能达到的最高密度尺度，$p_i\in[0,1]$。

最终我们将成员权重最大的选为代表元
$$s_C=
\arg\max_{y_i\in C}p_i.$$
请注意，我们这里取出的 $s_C$ 是最大成员权重的成员的索引。我们利用这个索引找到原始聚类代表元
$$z_{s_C}$$
这是我们最终选出的聚类代表元素。

问题又来了，我们可以发现一件事：HDESCAN 聚类的最终聚类代表元个数并不确定，我们不能像 K-means 那样直接指定我们需要多少个聚类，这意味着我们无法直接保证 IPC 就是聚类个数这件事。通常我们需要聚类个数恰好对齐每一类别最终蒸馏数据集个数，这样可以简洁地给出蒸馏数据集。

假设我们得到了代表样本
$$S=\{s_1,s_2,\ldots,s_M\}.$$
我们希望将 $M$ 调整为 IPC，这部分被称为 IPC Matching。

如果 $M\geq \mathrm{IPC}$，这非常容易处理，我们只需要选出最大的前 IPC 个聚类的代表元素。此处最大指的是包含最多节点，这意味着其密度更高并且代表了更加普遍的模式。

如果 $M \lt \mathrm{IPC}$，这会棘手一些，因为我们需要选出更多的代表元素。

第一种策略是 SplitCluster，简而言之就是我们将一些母聚类继续分裂产生更多聚类。实际上就是在母聚类内部再次进行 HDBSCAN 聚类。如果我们能够找到符合要求的两种子聚类，我们将这两个子聚类代表元素作为 K-means 中心计算 Inertia
$$\operatorname{Inertia}
=
\sum_{x\in c_{\mathrm{new1}}}
\|x-\mu_1\|_2^2
+
\sum_{x\in c_{\mathrm{new2}}}
\|x-\mu_2\|_2^2.$$
如果存在更多的两个子聚类这种 pair，我们比较他们的 Inertia 并且挑出 Inertia 更小的两个子聚类作为最后的挑选结果。

但是能够找出两个子聚类已经很乐观，如果始终无法找到符合要求的聚类，我们更换母聚类或者退化到第二种策略。

第二种策略是 Clustering Outliers。我们考虑 Outliers，也就是那些没有被选入最终任何聚类中的点
$$\mathcal O=\{o_1,o_2,\ldots,o_L\}.$$
我们缺少样本
$$q=\mathrm{IPC}-|M|$$
那么我们直接对 Outliers 做 K-means
$$K=q.$$
这样可以很简单地补充代表点。

第三种策略是 ForcedSplit，操作起来会更简单。这种策略是建立在 SplitCluster 之上的。当我们发现无法找到符合要求两个子聚类，就直接降低聚类大小门槛，这样可以使用更多点集合有资格成为聚类。

总之无论如何，聚类数量超过 IPC 都是容易处理的，但是聚类比 IPC 更少就需要额外分割聚类，以下是作者设计的 IPC Matching 的算法。

$$
\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Post-processing Scheme for IPC Matching} \\
\hline
\begin{aligned}
1: & \ \textbf{Global variables: } C, \text{ current clusters}; \, S, \text{ corresponding representative samples.} \\
2: & \ \textbf{procedure } \text{SPLITCLUSTER}(c_{\text{mother}}, \, min\_cluster\_size) \\
3: & \ \quad \text{Run HDBSCAN on } c_{\text{mother}} \text{ with } min\_cluster\_size \text{ to find } k \text{ sub-clusters as } S_{\text{cand}} \\
4: & \ \quad \textbf{If } k < 2 \textbf{ return False} \\
5: & \ \quad S \leftarrow S \setminus \{s_{\text{mother}}\} \textbf{ and } C \leftarrow C \setminus \{c_{\text{mother}}\} \\
6: & \ \quad \text{Iterate all pairs } (s_i, s_j) \text{ from } S_{\text{cand}} \text{ as initial seeds for K-Means } (k = 2) \text{ on } c_{\text{mother}}, \text{ select the} \\
   & \ \quad \text{pair yielding the minimum resulting inertia, denoted as } \{c_{\text{new1}}, c_{\text{new2}}\} \text{ and } \{s_{\text{new1}}, s_{\text{new2}}\} \\
7: & \ \quad C \leftarrow C \cup \{c_{\text{new1}}, c_{\text{new2}}\} \textbf{ and } S \leftarrow S \cup \{s_{\text{new1}}, s_{\text{new2}}\} \\
8: & \ \quad \textbf{return True} \\
9: & \ \textbf{end procedure} \\
\\
10:& \ \textbf{procedure } \text{FORCEDSPLIT}(c_{\text{mother}}, \, min\_size) \\
11:& \ \quad \textbf{while } \text{SPLITCLUSTER}(c_{\text{mother}}, \, min\_size) = \textbf{False} \ \& \ min\_size > 2 \textbf{ do} \\
12:& \ \quad\quad min\_size \leftarrow \max(2, \, \lfloor min\_size \times 0.75 \rfloor) \quad \quad \quad \quad \quad \quad \quad \quad \quad \triangleright \text{Relax the constraint} \\
13:& \ \quad \textbf{end while} \\
14:& \ \textbf{end procedure}
\end{aligned} \\
\hline
\end{array}
$$

我需要提醒，即使第一阶段 Distribution Discover 看起来非常繁琐复杂，其实我们仅仅只做了一件事，那就是找出一个合理的原始数据聚类。这个聚类原本是指 K-means，但是现在我们将其改为 HDBSCAN 聚类。

更多的，聚类方法的替换是完全 plug-in 的。我们还可以尝试更多聚类方法，比如 OPTICS 聚类或者 DBSCAN 聚类，只需要解决两个问题：第一，VAE latent 形状分布比较扭曲，无法应用球形聚类假设；第二，聚类数量最好严格等于 IPC，否则我们需要为剩余的 IPC 发明新的补充方法。

现在我们开始讲述第二阶段的 Distribution Alignment。这部分会简单很多，实际上和 MGD3 处理方式几乎一模一样，仅仅存在参数化上的区别。

## Distribution Alignment

我们已经从第一阶段得到了 IPC 个数的代表元素
$$S_r=\{s_1,s_2,\ldots,s_{\mathrm{IPC}}\}.$$
那么我们希望在生成第 $j$ 张图时朝着代表元素 $s_j$ 对齐。

实际上这里要做的事情和之前基于 Guidance 的数据集蒸馏方法一模一样，因为 CoDA 还是一种基于 Guidance 数据集蒸馏方法。

首先采样高斯噪声。对于第 $j$ 个生成图的每一个时间步 $t$ 时刻图像做如下几件事。

将图像按照时间步 $t$ 反推回到 clean point
$$\hat z_0(z_t)
=
\frac{1}{\sqrt{\bar\alpha_t}}
\left(
z_t
-
\sqrt{1-\bar\alpha_t}\,
\epsilon_\theta(z_t,t,c)
\right).$$
我们定义 Guidance 向量
$$g(z_t)
=
s_j-\hat z_0(z_t).$$
这个 Guidance 非常直接，我们希望指出 clean point 对于聚类中心的距离，这个 Guidance 实际上和 MGD3 使用的 Guidance 完全一模一样。

我们用一个超参数 $\gamma$ 控制对齐强度
$$\Delta \hat z_0
=
\gamma g(z_t)
=
\gamma
\left(
s_j-\hat z_0(z_t)
\right).$$
新的 clean point 估计是
$$\hat z_{0,\mathrm{new}}
=
\hat z_0+\Delta \hat z_0.$$
$$\hat z_{0,\mathrm{new}}
=
\hat z_0
+
\gamma(s_j-\hat z_0)
=
(1-\gamma)\hat z_0+\gamma s_j.$$
所以这本质上是一个原始预测 clean point 与聚类中心之间的线性插值结果。

更多的，存在关系
$$\Delta\epsilon_\theta
=
-
\frac{\sqrt{\bar\alpha_t}}{\sqrt{1-\bar\alpha_t}}
\Delta\hat z_0.$$
那么我们可以直接写出原始噪声预测与 Guidance 向量之间关系
$$\Delta\epsilon_\theta
=
\gamma\cdot g(z_t)
\cdot
\left(
-\frac{\sqrt{\bar\alpha_t}}{\sqrt{1-\bar\alpha_t}}
\right).$$

但是请注意，由于 CFG 的存在，我们只会对有条件部分加上 $\Delta\epsilon_\theta$，也就是说
$$\epsilon_\theta'(z_t,t,c)
=
\epsilon_\theta(z_t,t,c)
+
\Delta\epsilon_\theta.$$
那么 CFG 就是
$$\epsilon_{\mathrm{CFG}}
=
\epsilon_\theta(z_t,t,\varnothing)
+
w
\left(
\epsilon_\theta'(z_t,t,c)
-
\epsilon_\theta(z_t,t,\varnothing)
\right).$$
$$\epsilon_{\mathrm{CFG}}
=
\epsilon_{\mathrm{CFG,original}}
+
w\Delta\epsilon_\theta.$$

我们可以解释一下为什么不改动无条件生成部分。我们认为无条件生成部分代表了 Diffusion 模型对于什么是真实图像的先验判断，这是不需要聚类中心牵引的，只有有条件部分才会带判断地参与生成。

总结一下，CoDA 完全不是一个那么复杂的方法，其实就是将 MGD3 聚类改为 HDBSCAN 聚类。但是作者认为，就是这样一个简单的改动，可以让通用 T2I 具备某个狭窄数据集上的数据集蒸馏能力。

# 成果与讨论

CoDA 比对了基于 ImageNet 预训练生成模型的方法，比如 MGD3 等等方法的效果。在 ImageIDC 和 ImageNette 上，CoDA 超越了 MGD3，并且在 ImageNet-A 到 ImageNet-E 上对于架构泛化性的测试中，比如对于 ViT 与 ResNet 等等架构的比较，CoDA 方法达到了新的 State-of-the-Art。

更多的，CoDA 验证了一件事，那就是 Diffusion 模型确实起到了至关重要的作用。如果我们直接将 Distribution Discovery 的结果作为蒸馏数据集，效果完全没有原始方法好。这证明我们确实在利用通用 T2I 的先验知识。

我们之前很少探讨基于 Diffusion 的数据集蒸馏方法减少声称步数会如何，CoDA 的结论是，在减少生成步数到 $25$ 步情况下，CoDA 保持了出色的性能表现。

CoDA 实验中做的一个直观消融实验是，将 MGD3 各个部件逐渐换为 CoDA 部件，以证明后者效果。最终实验结果如下图所示。其中，MGD3(a) 是指原始 MGD3 方法，SDXL 指的是直接用 SDXL 按照类对应 Prompt 输出作为蒸馏数据集，MGD3(b) 指的是使用 SDXL 的 MGD3，MGD3(c) 指的是在 K-means 之前加上 UMAP 的 MGD3，MGD3(d) 指的是使用完整的 CoDA discovery 流程的 MGD3，以及 CoDA 本体。

<img src="./assets/CoDAabl.png" width="800" height="400">

可以看到每个组件都有其价值。

但是我想警醒一点，作者声称解决了通用 T2I 生成模型在狭窄数据集蒸馏上的诸多问题，真的如此吗？

我们来看一个极端例子：在我特制的数据集 A 中，所有狗都会穿戴卡通人物 Mr.Neverseen 图样的服装，其中 Mr.Neverseen 是一个我创造的并且从未在其他地方发布的卡通形象。也就是说在 A 中，图像与任何通用 T2I 之间会产生巨大的语义鸿沟。对于通用 T2I，狗大多是无服饰形象，但是现在却被要求蒸馏出一些穿戴特定图案服饰的狗。即使其受到聚类中心牵引，这个生成还是困难的。

所以我个人认为，CoDA 实际上某种程度上解决了通用 T2I 对于狭窄数据集蒸馏问题，但是严重依赖狭窄数据集本身也一定程度遵从真实数据集分布假设。在其实验中，ImageNette 和 ImageIDC 还是 ImageNet-1K 的子集，这是恰好符合这个假设的。CoDA 解决的其实是语义大体一致时，通用 T2I 先验与目标数据子分布之间的偏差。

所以我更愿意的叙述是，CoDA 还是一种基于 Guidance 的数据集蒸馏方法。其提出将聚类方法改为 HDBSCAN 密度聚类，发现能够显著改善数据集蒸馏中的分类难题。更多的，不仅仅对于专有训练在被蒸馏数据集上的模型，我们发现 CoDA 对于通用 T2I 模型也表现良好。

# 总结

本章介绍了 CoDA，核心要素是将 MGD3 的 K-means 聚类改为 HDBSCAN 聚类。

下一章我想谈谈 DMD，我们试图加速 Diffusion 蒸馏数据集过程中的生成步骤。